# Annotation

`docker pull alexanrna/scanpy-r:1.10-v6`

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import numpy as np 
from scipy import stats
import glob 
import os
from matplotlib import pyplot as plt
import datetime
import anndata as ad
from scipy import sparse
from anndata import AnnData

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)


In [ ]:
path = '/hdf5/20250220_all_data_integrated_filtered_umap_scANVI_metadata.cg.h5ad'
adata = sc.read_h5ad(path)

In [ ]:
adata

In [ ]:
adata.obs

In [ ]:
adata.X = adata.layers['log1p_norm']

In [ ]:
res = 2.5
sc.tl.leiden(adata, resolution=res, key_added="leiden_" + str(res))

In [ ]:
sc.pl.umap(adata, color = ['leiden_2.5','cell_type_scANVI'], ncols = 1, wspace=0.25)

In [ ]:
# compare with older annotation
old_path = '../01_processing_10x/hdf5/20241112_all_10x_cg_rna_filtered_umap_tsne_metadata.h5ad'
old = sc.read_h5ad(old_path)

In [ ]:
old

In [ ]:
old.obs.set_index('new_index', inplace=True)

In [ ]:
adata.obs['cell_type_majority_old'] = adata.obs.index.map(old.obs['cell_type_majority'])

In [ ]:
adata.obs['cell_type_majority_old'].value_counts()

In [ ]:
sc.pl.umap(adata, color = ['leiden_2.5','cell_type_majority_old','cell_type_scANVI'], ncols = 1, wspace=0.25)

## L4 IT neurons

In [ ]:
sc.pl.umap(adata, color = ['cell_type_majority_old'], groups = 'L4 IT',ncols = 1, wspace=0.25)

In [ ]:
sc.pl.umap(adata, color = ['leiden_2.5'])

In [ ]:
sc.pl.umap(adata, color = ['leiden_2.5'], groups = '23',ncols = 1, wspace=0.25)

In [ ]:
adata.obs['cell_type'] = adata.obs['cell_type_scANVI']

In [ ]:
def check_columns(row):
    if (row['leiden_2.5']=='23'):
        return 'L4 IT'
    else:
        return row['cell_type']


adata.obs['cell_type'] = adata.obs.apply(lambda row: check_columns(row), axis=1)

## L5 IT A - L5 IT B 

In [ ]:
sc.pl.umap(adata, color = ['cell_type_majority_old'], groups = 'L5 IT',ncols = 1, wspace=0.25)

In [ ]:
sc.pl.umap(adata, color = ['cell_type_majority_old'], groups = 'L5 IT A',ncols = 1, wspace=0.25)

In [ ]:
sc.pl.umap(adata, color = ['cell_type_majority_old'], groups = 'L5 IT B',ncols = 1, wspace=0.25)

In [ ]:
sc.pl.umap(adata, color = ['leiden_2.5'], groups = ['26','32','49'],ncols = 1, wspace=0.25)

In [ ]:
def check_columns(row):
    if (row['leiden_2.5'] in ['26','32','49']):
        return 'L5 IT A'
    else:
        return row['cell_type']


adata.obs['cell_type'] = adata.obs.apply(lambda row: check_columns(row), axis=1)

In [ ]:
sc.pl.umap(adata, color = ['leiden_2.5'], groups = ['3'],ncols = 1, wspace=0.25)

In [ ]:
def check_columns(row):
    if (row['leiden_2.5'] in ['3']):
        return 'L5 IT B'
    else:
        return row['cell_type']


adata.obs['cell_type'] = adata.obs.apply(lambda row: check_columns(row), axis=1)

## Immune

In [ ]:
sc.pl.umap(adata, color = 'cell_type_majority_old', groups='immune')

In [ ]:
sc.pl.umap(adata, color = ['leiden_2.5'], groups = '58',ncols = 1, wspace=0.25)

In [ ]:
def check_columns(row):
    if (row['leiden_2.5'] in ['58']):
        return 'immune'
    else:
        return row['cell_type']


adata.obs['cell_type'] = adata.obs.apply(lambda row: check_columns(row), axis=1)

In [ ]:
sc.pl.umap(adata, color = 'cell_type')

In [ ]:
sc.pl.umap(adata, color = 'cell_type', groups = 'L5 IT')

In [ ]:
import math
# code adapted from: https://github.com/aertslab/scATAC-seq_benchmark/blob/main/fixedcells_3_cistopic_consensus/5a_dimreduc.ipynb
clustering='leiden_2.5'
ct_pred_thr=0.95

to_plot = [clustering, 'consensus_cell_type', "cell_type"]
n_to_plot = len(to_plot)
n_cols = 3
n_rows = math.ceil(n_to_plot/n_cols)

clust_consensus = []
write = False

major_cell_type = []
frac_of_cluster = []
next_celltype = []
next_frac_of_cluster = []
diff_to_next = []
        
celldata = adata.obs
        
for c in celldata[clustering].unique():
    
    c1 = celldata[celldata[clustering]==c]
            # find proportions of each unique cell type detected in this cluster:
    ctp = pd.DataFrame([
    c1['cell_type'].unique(),
        [ c1[c1['cell_type']==x].shape[0] / c1.shape[0] for x in c1['cell_type'].unique() ]
        ]).T
            # sort/rank by proportion:
    ctps = ctp.sort_values(1,ascending=False)
    if ctp.shape[0]>1:
            n = ctps.iloc[1,0] # next cell type detected
            dtn = ctps.iloc[0,1] - ctps.iloc[1,1] # distance to next
            dtnf = ctps.iloc[1,1]
    else:
            dtn = None
            dtnf = None
            n = None
    major_cell_type.append(ctps.iloc[0,0])
    frac_of_cluster.append(ctps.iloc[0,1])
    next_celltype.append(n)
    next_frac_of_cluster.append(dtnf)
    diff_to_next.append(dtn)

    # collect results for this sample
    
    clust_consensus = pd.DataFrame([
        celldata[clustering].unique(),
        major_cell_type,
        frac_of_cluster,
        next_celltype,
        next_frac_of_cluster,
        diff_to_next,
            ]).T.set_axis(
                ['cluster', 'major_cell_type', 'frac_of_cluster', 
                 'next_celltype', 'next_frac_of_cluster', 'diff_to_next'],
                axis=1
            ).set_index(celldata[clustering].unique().astype(str))
   # print(sample)
    clust_consensus['cluster_counts'] = clust_consensus['cluster'].map(celldata[clustering].value_counts())
        # print only rows where there is another cell type within 20%:
    if(sum(clust_consensus['diff_to_next']<0.20)>0):
        display(clust_consensus[ clust_consensus['diff_to_next']<0.20 ])
            
    else:
        print("No discrepant clusters")
            
        # add the mapping
    adata.obs['consensus_cell_type'] = ""
    for i,r in clust_consensus.iterrows():
        ix = adata.obs[clustering] == r['cluster']
        adata.obs.loc[ix,'consensus_cell_type'] = r['major_cell_type']
 



In [ ]:
sc.pl.umap(adata, color = 'cell_type', groups = 'L5 IT')

In [ ]:
sc.pl.umap(adata, color = ['cell_type','consensus_cell_type'], ncols = 1)

In [ ]:
adata.obs['consensus_cell_type'] = adata.obs['consensus_cell_type'].apply(lambda x: 'L5 IT B' if x == 'L5 IT' else x)
sc.pl.umap(adata, color = ['cell_type','consensus_cell_type'], ncols = 1)

In [ ]:
sc.pl.umap(adata, color = ['batch'], ncols = 1)

In [ ]:
sc.pl.umap(adata, color = ['vireo_assignment']) #,groups = 'ASA_035', ncols = 1)

In [ ]:
adata.obs['consensus_cell_type'].value_counts()

In [ ]:
sc.pl.umap(adata, color = 'RORB')

In [ ]:
# add more broad annotation
cluster2annotation = {
    "L2/3 IT":"ExN",
    "Astro":"Astro",
    "Oligo":"Oligo",
    "OPC":"OPC",
    "Micro-PVM":"Micro-PVM",
    "Pvalb":"InN",
    "L5 IT B":"ExN",
    "Vip":"InN",
    "Lamp5":"InN",
    "Sst":"InN",
    "L5 IT A":"ExN",
    "L6 IT":"ExN",
    "Sncg":"InN",  
"L4 IT":"ExN",          
"L6 CT":"ExN",          
"L6b":"ExN",             
"L5/6 NP":"ExN",         
"L5 ET":"ExN",           
"L6 IT Car3":"ExN",  
"VLMC":"Endo",            
"Endo":"Endo",            
"immune":"Micro-PVM",           
"Sst Chodl":"InN",          
}

In [ ]:
adata.obs["cell_type_broad"] = (
    adata.obs["consensus_cell_type"].map(cluster2annotation).astype("category")
)

In [ ]:
adata.obs["consensus_cell_type"] = adata.obs["consensus_cell_type"].str.replace(r'[/\s]', '_', regex=True)

In [ ]:
adata.obs

In [ ]:
sc.pl.umap(adata, color = ['consensus_cell_type','cell_type_broad'], ncols = 1)

In [ ]:
date = datetime.datetime.now().strftime('%Y%m%d')

adata.write(f'./hdf5/{date}_all_data_integrated_filtered_umap_scANVI_metadata_ct_corrected.cg.h5ad')

In [ ]:
sc.pl.umap(adata, color = ['region'])

In [ ]:
grouped = adata.obs.groupby(['vireo_assignment', 'batch']).size().reset_index(name='counts')

# Create a dictionary to store the combined values
combined_dict = {}

# Determine the combined value for each donor
for donor, group in grouped.groupby('vireo_assignment'):
    if group['counts'].gt(0).sum() > 1:
        combined_dict[donor] = 'mix'
    else:
        combined_dict[donor] = group.loc[group['counts'].gt(0), 'batch'].values[0]

# Map the combined values back to the original DataFrame
adata.obs['combined'] = adata.obs['vireo_assignment'].map(combined_dict)